# Bài 1: Mạng LeNet — Tập dữ liệu MNIST
Xây dựng kiến trúc **LeNet**, huấn luyện với **Adam optimizer** và đánh giá trên các độ đo
**Accuracy, Precision (macro), Recall (macro), F1-macro**.

In [1]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tqdm import tqdm

os.makedirs("./checkpoints/assignment-1", exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

Device: cpu


In [2]:
class LeNet(nn.Module):
    def __init__(self):
        super().__init__()
        # Feature extractor
        self.conv1  = nn.Conv2d(1,  6,   kernel_size=5, padding=2)  # (B,1,28,28) -> (B,6,28,28)
        self.pool1  = nn.AvgPool2d(kernel_size=2, stride=2)          # -> (B,6,14,14)
        self.conv2  = nn.Conv2d(6,  16,  kernel_size=5)              # -> (B,16,10,10)
        self.pool2  = nn.AvgPool2d(kernel_size=2, stride=2)          # -> (B,16,5,5)
        self.conv3  = nn.Conv2d(16, 120, kernel_size=5)              # -> (B,120,1,1)
        # Classifier
        self.fc     = nn.Linear(120, 84)
        self.output = nn.Linear(84,  10)

    def forward(self, x):
        x = self.pool1(F.sigmoid(self.conv1(x)))
        x = self.pool2(F.sigmoid(self.conv2(x)))
        x = F.sigmoid(self.conv3(x)).view(x.size(0), -1)
        x = F.sigmoid(self.fc(x))
        return self.output(x)

model = LeNet()
print(model)

LeNet(
  (conv1): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
  (pool1): AvgPool2d(kernel_size=2, stride=2, padding=0)
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (pool2): AvgPool2d(kernel_size=2, stride=2, padding=0)
  (conv3): Conv2d(16, 120, kernel_size=(5, 5), stride=(1, 1))
  (fc): Linear(in_features=120, out_features=84, bias=True)
  (output): Linear(in_features=84, out_features=10, bias=True)
)


In [3]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_set = datasets.MNIST(root="./data", train=True,  download=True, transform=transform)
test_set  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_set,  batch_size=32, shuffle=False)

print(f"Train : {len(train_set):,} samples")
print(f"Test  : {len(test_set):,} samples")

100%|██████████| 9.91M/9.91M [00:00<00:00, 39.5MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.17MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 9.31MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 3.39MB/s]

Train : 60,000 samples
Test  : 10,000 samples


In [4]:
def evaluate(model, loader, device):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for images, labels in loader:
            out = model(images.to(device)).argmax(dim=-1).cpu().numpy()
            preds.extend(out)
            trues.extend(labels.numpy())
    return {
        "accuracy":  accuracy_score(trues, preds),
        "precision": precision_score(trues, preds, average="macro", zero_division=0),
        "recall":    recall_score   (trues, preds, average="macro", zero_division=0),
        "f1":        f1_score       (trues, preds, average="macro", zero_division=0),
    }

In [5]:
model     = LeNet().to(device)
loss_fn   = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

EPOCHS   = 10
best_f1  = 0.0

for epoch in range(1, EPOCHS + 1):
    model.train()
    losses = []
    with tqdm(train_loader, desc=f"[{epoch:02d}/{EPOCHS}] Training", ncols=90) as pb:
        for images, labels in pb:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(images), labels)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
            pb.set_postfix(loss=f"{np.mean(losses):.4f}")

    s = evaluate(model, test_loader, device)
    print(f"  Accuracy={s['accuracy']:.4f} | Precision={s['precision']:.4f} | Recall={s['recall']:.4f} | F1={s['f1']:.4f}")

    if s["f1"] > best_f1:
        best_f1 = s["f1"]
        torch.save(model.state_dict(), "./checkpoints/assignment-1/best_model.pth")

print(f"\nBest F1: {best_f1:.4f}")

[01/10] Training: 100%|██████████████████| 1875/1875 [00:43<00:00, 42.73it/s, loss=0.6383]


  Accuracy=0.9377 | Precision=0.9386 | Recall=0.9368 | F1=0.9371


[02/10] Training: 100%|██████████████████| 1875/1875 [00:42<00:00, 43.81it/s, loss=0.1604]


  Accuracy=0.9649 | Precision=0.9650 | Recall=0.9647 | F1=0.9647


[03/10] Training: 100%|██████████████████| 1875/1875 [00:42<00:00, 44.03it/s, loss=0.1045]


...

  Accuracy=0.9839 | Precision=0.9839 | Recall=0.9839 | F1=0.9838


[09/10] Training: 100%|██████████████████| 1875/1875 [00:42<00:00, 44.27it/s, loss=0.0371]


  Accuracy=0.9848 | Precision=0.9850 | Recall=0.9846 | F1=0.9848


[10/10] Training: 100%|██████████████████| 1875/1875 [00:43<00:00, 43.37it/s, loss=0.0339]


  Accuracy=0.9874 | Precision=0.9874 | Recall=0.9873 | F1=0.9873

Best F1: 0.9873


In [6]:
model.load_state_dict(torch.load("./checkpoints/assignment-1/best_model.pth", map_location=device))
s = evaluate(model, test_loader, device)

for k, v in s.items():
    print(f"  {k.capitalize():<12}: {v:.4f}")

  Accuracy    : 0.9874
  Precision   : 0.9874
  Recall      : 0.9873
  F1          : 0.9873
